# aw_03_c — Stages C1/C2: instruct reference baselines (G4 anchor)

**Protocol**: §4 reference baselines, §5.1 C1/C2. No training — two eval runs of
Qwen/Qwen3-8B (post-trained), pinned revision `b968826d…`.

- **C1**: zero-shot, no opener seed (the instruct model writes its own think block).
- **C2**: few-shot k=3 with FROZEN exemplars drawn from train families only
  (`scripts/build_fewshot_exemplars.py`) — leakage gate preserved.

Suites, verifier, and the greedy decoding profile are identical to A1/base evals;
only the conditioning profile (`evaluation:` config block) differs, per amendment v1.1.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


Cloning into 'axiom-world'...
remote: Enumerating objects: 421, done.
remote: Counting objects: 100% (421/421), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 421 (delta 209), reused 335 (delta 123), pack-reused 0 (from 0)
Receiving objects: 100% (421/421), 154.00 KiB | 6.16 MiB/s, done.
Resolving deltas: 100% (209/209), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 196.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 74.3 MB/s eta 0:00:00
  Building editable for axiom-world (pyproject.toml) ... done

In [ ]:
# @title a_c_data — frozen suites + frozen few-shot exemplars
!python scripts/build_eval_suites.py --episodes-per-suite 300
!python scripts/build_training_data.py
!python scripts/build_fewshot_exemplars.py \
  --sft-file data/train/playworld_sft.jsonl \
  --output data/eval_suites/fewshot_exemplars.json \
  --count 3


eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training loaders must pass eval_family_ids as forbidden_family_ids (leakage gate).
{
  "seed": 1042,
  "sft_records": 2000,
  "prompt_records": 2000,
  "unsolvable_dropped": 0,
  "sft_fingerprint": "sha256:90d42af5b30dd0b6c128cbc552d891f9d3a7b4f36c27b123cecd1a50f99a64cc",
  "prompt_fingerprint": "sha256:cc2aef0df4f5efda3db7ccfd43c76e40260935e9b7b55ee5760490e6a4304f14",
  "train_families": [
    "train-fam0",
    "train-fa

In [ ]:
# @title c_c1_eval — instruct zero-shot
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld_c1.yaml \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
config.json: 100% 728/728 [00:00<00:00, 7.97MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 27.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 13.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 15.0MB/s]

tokenizer.json: downloading bytes:   0% 13.4k/11.4M [00:00<08:12, 23.2kB/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:00<00:00, 4.58MB/s,  334kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:00<00:00, 15.4MB/s, 1.12MB/s  ]
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 34.2MB/s]
Reconstructing (incomplete total...): |     

In [ ]:
# @title d_c2_eval — instruct few-shot (k=3, frozen exemplars)
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld_c2.yaml \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 339.80it/s]
generate(batched): 100% 3/3 [13:37<00:00, 272.41s/it]
eval_adversarial: pass_rate={'mean': 0.9067, 'ci95': [0.8733, 0.94]}
generate(batched): 100% 3/3 [13:56<00:00, 278.84s/it]
eval_comp_ood: pass_rate={'mean': 0.1367, 'ci95': [0.1, 0.1767]}
generate(batched): 100% 3/3 [13:53<00:00, 277.94s/it]
eval_id: pass_rate={'mean': 0.2267, 'ci95': [0.18, 0.2733]}
generate(batched): 100% 3/3 [13:57<00:00, 279.22s/it]
eval_rule_ood: pass_rate={'mean': 0.1367, 'ci95': [0.1, 0.1767]}
generate(batched): 100% 3/3 [13:55<00:00, 278.59s/it]
eval_

In [ ]:
# @title f_c_analysis — anchor comparisons
RUN_ID_a1   = "20260801-063425--eval-playworld--s42--3bf440"  # A1 eval run
RUN_ID_c1   = "20260802-053440--eval-playworld-c1-instruct-zeroshot--s42--e6cb05"  # <- fill from c_c1_eval output ("eval run: ...")
RUN_ID_c2   = "20260802-060802--eval-playworld-c2-instruct-fewshot--s42--67cf20"  # <- fill from d_c2_eval output

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_a1} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_c1} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_c2} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{RUN_ID_a1} --label-a a1-sft \
  --run-b runs/{RUN_ID_c1} --label-b qwen3-8b-instruct-0shot \
  --output runs/{RUN_ID_a1}/analysis_vs_c1.json --hf-sync-repo m97j/aw-runs-a1

!python scripts/run_analysis.py \
  --run-a runs/{RUN_ID_a1} --label-a a1-sft \
  --run-b runs/{RUN_ID_c2} --label-b qwen3-8b-instruct-3shot \
  --output runs/{RUN_ID_a1}/analysis_vs_c2.json --hf-sync-repo m97j/aw-runs-a1

eval run materialized: 5 suite files, freeze_fingerprint=sha256:3cdcbc30c99e492c...
RUN_DIR=runs/20260801-063425--eval-playworld--s42--3bf440
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.18M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/2.37M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/2.37M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 540/3.56M [00:00<30:22, 1.95kB/s]
Reconstructing (incomplete total...):   0% 540/4.75M [00:00<40:33, 1.95kB/s]
Reconstructing (incomplete total...):   0% 540/5.92M [00:00<50:34, 1.95kB/s]
Reconstructing (incomplete total...):   0% 540/5.92M [00:00<50:34, 1.95kB/s]

Fetching 8 files:  12% 1/8 [00:00<00:02,  3.24it/s]
Reconstructing (incomplete total...):   0% 1.12k/5.93M [00:00<50:35, 1.95kB/s]

Fetching 8 files: 100% 8/8 [00:00<00:00, 17.00it/s]
Download complete: 100

## Gate checklist (G4)
- [x] C1/C2 summaries persisted to the Hub
- [x] A1 vs C1 / A1 vs C2 analysis synced
- [x] Reference anchors recorded — headline tables may now cite C1/C2
